In [1]:
import polars as pl
import leafmap.maplibregl as leafmap
import numpy as np
from matplotlib import pyplot as plt
from pathlib import Path
import sys

In [5]:
DATA_DIR = Path.cwd().parent / "data"
ICP_DIR = DATA_DIR / "ICP"
ICP_OBSERVATIONS = ICP_DIR / "icpf/03_tidy/cpf-level2_cleaned.parquet"

In [7]:
# load parquet file as dataframe
df = pl.read_parquet(
    ICP_OBSERVATIONS, columns=["plot_id", "plot_latitude", "plot_longitude", "growth_rate"]
)

# Group by plot (id, lat, long) -> mean growth per year
plots = df.group_by("plot_id", "plot_latitude", "plot_longitude").agg(
    pl.len().alias("n_periods"),
    pl.col("growth_rate").mean().alias("mean_growth_cm_year"),
)

In [8]:
# Transform degrees to decimal -> fit the leafmap and EarthEngine coord. system
def dms_to_decimal(val):
    degs, rem = divmod(abs(val), 10_000)
    mins, secs = divmod(rem, 100)
    sign = -1 if val < 0 else 1
    return sign * (degs + mins / 60 + secs / 3600)

In [9]:
# Setup the features for leafmap
features = []

for row in plots.iter_rows(named=True):
    lon = dms_to_decimal(row["plot_longitude"])
    lat = dms_to_decimal(row["plot_latitude"])

    # dict with leafmap format
    features.append(
        {
            "type": "Feature",
            "geometry": {"type": "Point", "coordinates": [lon, lat]},
            "properties": {
                "plot_id": row["plot_id"],
                "n_periods": row["n_periods"],
                "mean_growth_cm_year": row["mean_growth_cm_year"],
            },
        }
    )

In [ ]:
# Create and display the map
m = leafmap.Map(height="650px")
m.add_geojson(
    {"type": "FeatureCollection", "features": features},
    name="ICP plots",
    layer_type="circle",
    paint={
        "circle-radius": 5,
        "circle-stroke-color": "#ffffff",
        "circle-stroke-width": 1,
        "circle-color": [
            "interpolate",
            ["linear"],
            ["get", "mean_growth_cm_year"],
            0.15,
            "red",
            0.28,
            "violet",
            0.48,
            "blue",
        ],
    },
)
m

Html(children=[<leafmap.maplibregl.Map object at 0x00000144EB8C3BB0>, Card(children=[Btn(children=[Icon(childr…